# 02 — Exploratory Data Analysis

Explores the hourly table built by `01_preprocessing.ipynb` (run that first).

Questions:
1. How is Kp distributed, and how rare are storms?
2. What do the solar-wind inputs look like?
3. Where and how long are the data gaps?
4. Which inputs relate to Kp, and with what time delay?
5. How does activity follow the solar cycle?
6. What does a real extreme storm look like (May 2024)?
7. Are extreme values errors or real physics?

Figures are saved to `reports/figures/` for the report.

> Modelling decisions should rest on the **training years (1998–2016)**; test-period plots are descriptive only.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.preprocessing import load_processed

FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")

df = load_processed()
year = df.index.year
df["split"] = np.select([year <= config.TRAIN_YEARS[1], year <= config.VAL_YEARS[1]],
                         ["train", "val"], "test")
df["level"] = pd.cut(df["kp"], bins=config.KP_BINS, labels=config.KP_BIN_LABELS, right=False)
df["vbs"] = df["speed"] * (-df["bz_gsm"]).clip(lower=0) * 1e-3   # V·Bs, mV/m (rectified E-field)
train = df[df["split"] == "train"]
print(df.shape, df.index.min(), '→', df.index.max())

## 1. Target: Kp distribution and storm rarity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
edges = np.arange(0, 9 + 2/3, 1/3) - 1/6
axes[0].hist(df["kp"], bins=edges, color="#4C72B0")
axes[0].axvline(config.STORM_THRESHOLD, color="crimson", ls="--", label="storm threshold (Kp 5-)")
axes[0].set(title="Kp distribution (hourly rows)", xlabel="Kp", ylabel="hours", yscale="log")
axes[0].legend()

share = pd.crosstab(df["level"], df["split"], normalize="columns")[["train", "val", "test"]] * 100
share.plot.bar(ax=axes[1], rot=0)
axes[1].set(title="Storm-level share by split", xlabel="Kp level", ylabel="% of hours", yscale="log")
save(fig, "01_kp_distribution")
plt.show()

print(f"Storm hours (Kp ≥ 5-): {(df['kp'] >= config.STORM_THRESHOLD).mean():.2%} of all rows")
share.round(3)

## 2. Input distributions

In [ ]:
cols = [("b_scalar", "|B| (nT)", False), ("bz_gsm", "Bz GSM (nT)", False),
        ("speed", "Speed (km/s)", False), ("density", "Density (cm⁻³), log10", True),
        ("temperature", "Temperature (K), log10", True), ("f107_obs", "F10.7 (sfu)", False)]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (c, label, log) in zip(axes.flat, cols):
    s = df[c].dropna()
    s = np.log10(s[s > 0]) if log else s
    ax.hist(s, bins=80, color="#55A868")
    ax.set(title=label, ylabel="hours")
fig.tight_layout()
save(fig, "02_input_distributions")
plt.show()

df[[c for c, _, _ in cols]].describe().T.round(2)

## 3. Missing data: how much, when, and how long

In [ ]:
raw = config.RAW_FEATURES
miss_year = df[raw].isna().groupby(df.index.year).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 2]})
(df[raw].isna().mean() * 100).sort_values().plot.barh(ax=axes[0], color="#C44E52")
axes[0].set(title="Missing % (all years)", xlabel="%")
im = axes[1].imshow(miss_year.T, aspect="auto", cmap="Reds", vmin=0, vmax=max(10, miss_year.values.max()))
axes[1].set_yticks(range(len(raw)), raw)
axes[1].set_xticks(range(0, len(miss_year), 2), miss_year.index[::2], rotation=45)
axes[1].set_title("Missing % by year")
axes[1].grid(False)
fig.colorbar(im, ax=axes[1], label="%")
fig.tight_layout()
save(fig, "03_missingness")
plt.show()

In [ ]:
def gap_lengths(s):
    """Length (hours) of each run of consecutive NaNs."""
    isna = s.isna()
    runs = isna.ne(isna.shift()).cumsum()
    lengths = isna.groupby(runs).sum()
    return lengths[lengths > 0]

gaps = pd.DataFrame({c: gap_lengths(df[c]).describe(percentiles=[.5, .9, .99])
                     for c in ["bz_gsm", "speed", "density", "temperature"]}).T
for c in ["bz_gsm", "density"]:
    g = gap_lengths(df[c])
    print(f"{c}: {len(g)} gaps, {(g <= 3).mean():.0%} are ≤3 h, "
          f"hours in gaps >24 h: {g[g > 24].sum() / g.sum():.0%} of missing hours")
gaps.round(1)

## 4. Which inputs drive Kp?
Physics expectation: southward IMF (negative Bz) and fast wind open the magnetosphere to energy input, so Kp should rise as Bz goes negative and as V·Bs grows. Northward Bz should matter little.

In [ ]:
def binned(x, y, bins):
    g = y.groupby(pd.cut(x, bins), observed=True)
    out = g.quantile([.25, .5, .75]).unstack()
    out["n"] = g.size()
    out.index = [iv.mid for iv in out.index]
    return out[out["n"] >= 50]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (x, bins, label) in zip(axes, [
        (train["bz_gsm"], np.arange(-30, 31, 2), "Bz GSM (nT)"),
        (train["speed"], np.arange(250, 1001, 25), "Speed (km/s)"),
        (train["vbs"], np.arange(0, 20.5, 0.5), "V·Bs (mV/m)")]):
    b = binned(x, train["kp"], bins)
    ax.fill_between(b.index, b[0.25], b[0.75], alpha=0.3, label="25–75%")
    ax.plot(b.index, b[0.5], marker="o", ms=3, label="median Kp")
    ax.axhline(config.STORM_THRESHOLD, color="crimson", ls="--", lw=1)
    ax.set(xlabel=label, ylabel="Kp", ylim=(0, 9))
axes[0].legend()
fig.suptitle("Kp vs solar-wind drivers (training years)")
fig.tight_layout()
save(fig, "04_kp_vs_drivers")
plt.show()

In [ ]:
cands = raw + ["vbs"]
corr = train[cands].corrwith(train["kp"], method="spearman").sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
corr.plot.barh(ax=ax, color=["#C44E52" if v < 0 else "#4C72B0" for v in corr])
ax.set(title="Spearman correlation with Kp (training years)", xlabel="ρ")
save(fig, "05_correlation")
plt.show()
corr.round(3)

## 5. Time delay: how far back does the solar wind still matter?
Correlation of Kp with V·Bs, Bz and speed shifted *k* hours into the past. This justifies the lag and rolling-window features in `features.py`.

In [ ]:
lags = range(0, 25)
lagcorr = pd.DataFrame({c: [train["kp"].corr(train[c].shift(k), method="spearman") for k in lags]
                        for c in ["vbs", "bz_gsm", "speed"]}, index=lags)
fig, ax = plt.subplots(figsize=(8, 4))
lagcorr.abs().plot(ax=ax, marker="o", ms=3)
ax.set(title="|Spearman ρ| between Kp and driver k hours earlier", xlabel="lag k (hours)", ylabel="|ρ|")
save(fig, "06_lag_correlation")
plt.show()
lagcorr.iloc[[0, 1, 3, 6, 9, 12, 24]].round(3)

## 6. Solar cycle

In [ ]:
yr = df.groupby(df.index.year).agg(kp=("kp", "mean"), f107=("f107_obs", "mean"),
                                    storm=("kp", lambda s: (s >= config.STORM_THRESHOLD).mean() * 100))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(yr.index, yr["storm"], color="#C44E52", alpha=0.6, label="storm hours (%)")
ax.plot(yr.index, yr["kp"], color="k", marker="o", label="mean Kp")
ax.set(xlabel="year", ylabel="mean Kp  /  storm %")
ax2 = ax.twinx()
ax2.plot(yr.index, yr["f107"], color="#DD8452", ls="--", label="F10.7")
ax2.set_ylabel("F10.7 (sfu)")
ax2.grid(False)
for edge in [config.TRAIN_YEARS[1] + .5, config.VAL_YEARS[1] + .5]:
    ax.axvline(edge, color="grey", ls=":")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88))
ax.set_title("Yearly activity vs solar cycle (dotted: train | val | test)")
save(fig, "07_solar_cycle")
plt.show()
yr.round(2).T

## 7. Case study: May 2024 (Gannon) storm — held out in the test set

In [ ]:
ev = df.loc["2024-05-09":"2024-05-13"]
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, (c, label) in zip(axes, [("bz_gsm", "Bz (nT)"), ("speed", "V (km/s)"),
                                 ("density", "N (cm⁻³)"), ("kp", "Kp")]):
    if c == "kp":
        ax.step(ev.index, ev[c], where="post", color="crimson")
        ax.axhline(config.STORM_THRESHOLD, color="grey", ls="--", lw=1)
        ax.set_ylim(0, 9.3)
    else:
        ax.plot(ev.index, ev[c])
    ax.set_ylabel(label)
axes[0].axhline(0, color="grey", lw=1)
axes[0].set_title("May 2024 storm: solar wind arrives, Bz turns strongly south, Kp hits 9")
fig.tight_layout()
save(fig, "08_may2024_storm")
plt.show()

## 8. Extreme values: errors or real physics?
If extremes are real, they should cluster in storm periods rather than appear at random.

In [ ]:
pct = df[["b_scalar", "bz_gsm", "speed", "density", "temperature"]].quantile(
    [0, .001, .01, .5, .99, .999, 1]).T
display(pct.round(1))

checks = {"|Bz| > 30 nT": df["bz_gsm"].abs() > 30, "speed > 900 km/s": df["speed"] > 900,
          "density > 50 cm⁻³": df["density"] > 50, "temperature > 1e6 K": df["temperature"] > 1e6}
storm = df["kp"] >= config.STORM_THRESHOLD
pd.DataFrame({"hours": {k: m.sum() for k, m in checks.items()},
              "% during storms": {k: round(storm[m].mean() * 100, 1) for k, m in checks.items()},
              "baseline storm %": round(storm.mean() * 100, 1)})